# **Proyek ETL Pipeline Bank Marketing (Databricks Free Edition)**
## **Layer 2: Silver Layer — Cleansing, Standardization, Deduplication & Feature Engineering (Task 2 Workflow)**

Notebook ini berperan sebagai **Task 2 (`Clean_Silver`)** yang menunggu **Task 1 (`Ingest_Bronze`)** selesai.

### **Transformasi & Aturan Kualitas Data Layer Silver:**
1. **Standarisasi Tipe & Format:** Memastikan tipe numerik (`INT`) konsisten serta melakukan `trim` + `lower` pada seluruh kolom kategorikal.
2. **Penanganan Null & `"unknown"`:** Mengisi nilai `NULL` kategorikal menjadi `"unknown"`, membuang baris yang kehilangan kolom kunci wajib (`age`, `balance`, `day`, `month`, `y`), serta mencatat statistik nilai `"unknown"`.
3. **Filtering Validitas Domain:** Memfilter baris tidak valid di luar aturan bisnis (`age` 18–100, `day` 1–31, `duration >= 0`, `campaign >= 1`, `previous >= 0`, `month` valid 1–12, dan `y` bernilai `'yes'`/`'no'`).
4. **Deduplikasi Deterministik:** Menggunakan `Window` (`row_number`) berdasarkan seluruh atribut bisnis asli + `campaign_year`, mempertahankan catatan terbaru berdasarkan `ingested_at`.
5. **Feature Engineering:** Membuat `campaign_year`, `month_num`, `year_month`, `campaign_date` (`DATE`), `age_group`, `balance_segment`, `was_previously_contacted`, dan target biner `y_flag` (`1`/`0`).
6. **Penyimpanan Idempotent:** Menulis tabel **Managed Delta** `{catalog}.{schema}.{silver_table}` secara *overwrite* sehingga aman dijalankan ulang berkali-kali.

---
### **1. Parameter Task Workflow (`dbutils.widgets`)**

In [ ]:
import json
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

dbutils.widgets.text("catalog", "workspace", "Unity Catalog Name")
dbutils.widgets.text("schema", "bank_marketing", "Schema Name")
dbutils.widgets.text("bronze_table", "bronze_bank_marketing", "Nama Tabel Delta Bronze")
dbutils.widgets.text("silver_table", "silver_bank_marketing", "Nama Tabel Delta Silver")

catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
bronze_table = dbutils.widgets.get("bronze_table").strip()
silver_table = dbutils.widgets.get("silver_table").strip()

full_bronze_table = f"{catalog}.{schema}.{bronze_table}"
full_silver_table = f"{catalog}.{schema}.{silver_table}"

if not spark.catalog.tableExists(full_bronze_table):
    raise ValueError(
        f"Tabel Bronze {full_bronze_table} belum tersedia. "
        "Pastikan Task 1 (01_Bronze_Layer) berhasil dijalankan terlebih dahulu."
    )

print(f"Sumber Bronze : {full_bronze_table}")
print(f"Target Silver : {full_silver_table}")

---
### **2. Muat Data Bronze & Audit Kualitas Data Sebelum Pembersihan**

In [ ]:
df_bronze = spark.table(full_bronze_table)
rows_before = df_bronze.count()

if rows_before == 0:
    raise ValueError(f"Tabel Bronze {full_bronze_table} kosong (0 baris). Tidak ada data untuk diproses.")

CATEGORICAL_COLS = [
    "job", "marital", "education", "default",
    "housing", "loan", "contact", "month", "poutcome", "y"
]
NUMERIC_COLS = ["age", "balance", "day", "duration", "campaign", "pdays", "previous"]
RAW_FEATURE_COLS = [
    "age", "job", "marital", "education", "default", "balance",
    "housing", "loan", "contact", "day", "month", "duration",
    "campaign", "pdays", "previous", "poutcome", "y"
]
VALID_MONTHS = ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"]

print(f"Jumlah baris awal dari Bronze : {rows_before:,}")
print("\n=== Audit SQL NULL Sebelum Pembersihan ===")
display(df_bronze.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in RAW_FEATURE_COLS]))

print("=== Audit Nilai 'unknown' Sebelum Pembersihan ===")
display(df_bronze.select([
    F.sum((F.lower(F.trim(F.col(c))) == "unknown").cast("int")).alias(c)
    for c in ["job", "education", "contact", "poutcome"]
]))

---
### **3. Standarisasi Tipe, Penanganan Null, Filter Validitas & Deduplikasi**

In [ ]:
# 3A. Standarisasi string (trim + lower) & casting eksplisit tipe numerik
df_std = df_bronze
for c in CATEGORICAL_COLS:
    df_std = df_std.withColumn(c, F.lower(F.trim(F.col(c).cast(T.StringType()))))

for c in NUMERIC_COLS:
    df_std = df_std.withColumn(c, F.col(c).cast(T.IntegerType()))

# 3B. Penanganan Null
# Kolom kategorikal opsional diisi 'unknown', sedangkan kolom kunci wajib yang NULL di-drop
optional_cat_cols = ["job", "marital", "education", "default", "housing", "loan", "contact", "poutcome"]
df_null_handled = df_std.fillna("unknown", subset=optional_cat_cols)
df_null_handled = df_null_handled.dropna(subset=["age", "balance", "day", "month", "y"])

# 3C. Ekstrak campaign_year dari source_file secara aman untuk ANSI mode
year_str_expr = F.regexp_extract(F.col("source_file"), r"(\d{4})", 1)
df_with_year = df_null_handled.withColumn(
    "campaign_year",
    F.when(year_str_expr != "", year_str_expr.cast(T.IntegerType())).otherwise(F.lit(2008))
)

# 3D. Filter aturan validitas domain
df_valid = df_with_year.filter(
    (F.col("age").between(18, 100))
    & (F.col("day").between(1, 31))
    & (F.col("duration") >= 0)
    & (F.col("campaign") >= 1)
    & (F.col("pdays").isNotNull())
    & (F.col("pdays") >= -1)
    & (F.col("previous") >= 0)
    & (F.col("month").isin(VALID_MONTHS))
    & (F.col("y").isin("yes", "no"))
)
rows_after_filter = df_valid.count()
invalid_rows_removed = rows_before - rows_after_filter

# 3E. Deduplikasi deterministik berdasarkan kolom bisnis + tahun kampanye
dedup_subset = RAW_FEATURE_COLS + ["campaign_year"]
dedup_window = Window.partitionBy(*dedup_subset).orderBy(
    F.col("ingested_at").desc(),
    F.col("batch_id").desc(),
)
df_dedup = (
    df_valid
    .withColumn("_row_num", F.row_number().over(dedup_window))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num")
)
rows_after_dedup = df_dedup.count()
duplicate_rows_removed = rows_after_filter - rows_after_dedup

print(f"Baris setelah filter domain : {rows_after_filter:,} (dibuang: {invalid_rows_removed:,})")
print(f"Baris setelah deduplikasi   : {rows_after_dedup:,} (duplikat dibuang: {duplicate_rows_removed:,})")

---
### **4. Feature Engineering (Atribut Waktu, Segmen Bisnis & Target Numerik)**

In [ ]:
month_num_expr = (
    F.when(F.col("month") == "jan", 1)
    .when(F.col("month") == "feb", 2)
    .when(F.col("month") == "mar", 3)
    .when(F.col("month") == "apr", 4)
    .when(F.col("month") == "may", 5)
    .when(F.col("month") == "jun", 6)
    .when(F.col("month") == "jul", 7)
    .when(F.col("month") == "aug", 8)
    .when(F.col("month") == "sep", 9)
    .when(F.col("month") == "oct", 10)
    .when(F.col("month") == "nov", 11)
    .when(F.col("month") == "dec", 12)
    .otherwise(F.lit(None).cast(T.IntegerType()))
)

df_silver = (
    df_dedup
    .withColumn("month_num", month_num_expr)
    .withColumn(
        "year_month",
        F.concat(
            F.col("campaign_year").cast(T.StringType()),
            F.lit("-"),
            F.lpad(F.col("month_num").cast(T.StringType()), 2, "0"),
        ),
    )
    .withColumn("campaign_date", F.to_date(F.concat(F.col("year_month"), F.lit("-01")), "yyyy-MM-dd"))
    .withColumn(
        "age_group",
        F.when(F.col("age") < 30, "Muda (<30)")
        .when((F.col("age") >= 30) & (F.col("age") <= 50), "Paruh Baya (30-50)")
        .otherwise("Tua (>50)"),
    )
    .withColumn(
        "balance_segment",
        F.when(F.col("balance") < 0, "Negatif (<0)")
        .when(F.col("balance") <= 500, "Rendah (0-500)")
        .when(F.col("balance") <= 2000, "Menengah (501-2000)")
        .otherwise("Tinggi (>2000)"),
    )
    .withColumn("was_previously_contacted", F.when(F.col("pdays") != -1, 1).otherwise(0))
    .withColumn("y_flag", F.when(F.col("y") == "yes", 1).otherwise(0))
    .withColumn("silver_processed_at", F.current_timestamp())
)

print("Skema Silver Layer:")
df_silver.printSchema()

---
### **5. Tulis Tabel Delta Silver & Laporan Kualitas Data Sebelum/Sesudah**

In [ ]:
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(full_silver_table)
)

spark.sql(
    f"COMMENT ON TABLE {full_silver_table} IS "
    "'Silver Layer Bank Marketing: data bersih, terstandarisasi, terdeduplikasi, dengan fitur turunan waktu & segmentasi.'"
)

df_silver_saved = spark.table(full_silver_table)
rows_after = df_silver_saved.count()

print("=== LAPORAN KUALITAS DATA (SEBELUM vs SESUDAH SILVER) ===")
quality_summary = spark.createDataFrame(
    [
        ("1. Total baris awal (Bronze)", int(rows_before)),
        ("2. Baris invalid/null dibuang", int(invalid_rows_removed)),
        ("3. Baris duplikat dibuang", int(duplicate_rows_removed)),
        ("4. Total baris bersih akhir (Silver)", int(rows_after)),
    ],
    schema="tahap_kualitas_data STRING, jumlah_baris LONG",
)
display(quality_summary)

print("=== Verifikasi SQL NULL Setelah Silver (Harus 0) ===")
check_cols = RAW_FEATURE_COLS + ["campaign_year", "month_num", "year_month", "campaign_date", "age_group", "balance_segment", "y_flag"]
display(df_silver_saved.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in check_cols]))

print("=== Ringkasan Data Silver per Tahun Kampanye & Kelompok Usia ===")
display(
    df_silver_saved
    .groupBy("campaign_year", "age_group")
    .agg(
        F.count("*").alias("jumlah_klien"),
        F.sum("y_flag").alias("jumlah_subscribed"),
        F.round(F.sum("y_flag") * 100.0 / F.count("*"), 2).alias("conversion_rate_pct"),
    )
    .orderBy("campaign_year", "age_group")
)

result_payload = {
    "status": "SUCCESS",
    "table": full_silver_table,
    "rows_before": rows_before,
    "invalid_rows_removed": invalid_rows_removed,
    "duplicate_rows_removed": duplicate_rows_removed,
    "rows_after": rows_after,
}
dbutils.notebook.exit(json.dumps(result_payload))